# Chapter 3: Introduction to SQL


## Core Question

How can SQL define relational structure, retrieve the intended rows, handle duplicates
and `NULL`, summarize groups, and modify data without violating constraints?

Use this guide with the executable SQL lab cell. Before running it in a new SQLite database, run the
Chapter 2 the shared setup SQL cell file.


## Connection to Chapter 2

| Relational-algebra idea | Main SQL expression |
|---|---|
| Projection `Π` | Column list in `SELECT` |
| Selection `σ` | Predicate in `WHERE` |
| Cartesian product `×` | Multiple inputs without a matching predicate |
| Rename `ρ` | `AS` aliases |
| Union `∪` | `UNION` |
| Intersection `∩` | `INTERSECT` |
| Difference `−` | `EXCEPT` |

This table explains query meaning, not physical execution order. Chapters 15 and 16
address execution plans.


## Teaching Summary

| Topic | Worked example and practice | Evidence to retain |
|---|---|---|
| Table definition and basic queries | Course-registration tables and row filters | Executed SQL and checked rows |
| Duplicates, set operations, and `NULL` | `DISTINCT`, `EXCEPT`, and three-valued logic | Prediction and observed result |
| Grouping and subqueries | Department totals and nested enrollment queries | Intermediate and final results |
| Data modification | Reversible insert, update, and delete | Target-row check and rollback evidence |


## Prerequisites

- Identify relations, attributes, primary keys, and foreign keys.
- Apply selection, projection, product, and set operations to small relations.
- Predict whether a modification could violate a key or reference rule.


## Learning Objectives

After completing this chapter, you should be able to:

1. Create tables with data types, keys, `NOT NULL`, and simple `CHECK` constraints.
2. Use `SELECT`, `FROM`, `WHERE`, aliases, expressions, patterns, and ordering.
3. Distinguish default duplicate behavior from `DISTINCT` and SQL set operations.
4. Use `IS NULL` and explain why `UNKNOWN` does not pass `WHERE`.
5. Use aggregate functions, `GROUP BY`, and `HAVING`.
6. Use `IN`, `EXISTS`, one correlated subquery, and one CTE.
7. Explain the `NOT IN` risk when the comparison set contains `NULL`.
8. Predict and verify the effects of `INSERT`, `UPDATE`, and `DELETE`.

Before each example, predict the result columns, row count, duplicate or `NULL` behavior,
and whether the statement reads data, modifies data, or changes the schema.


## 1. Defining Structure

SQL includes data-definition statements, data-manipulation statements, and integrity
constraints. This course uses SQLite for executable work, so its examples use `TEXT` and
`INTEGER`. SQLite type affinity is not a rule for every DBMS.

```sql
CREATE TABLE study_group (
    group_id TEXT PRIMARY KEY,
    group_name TEXT NOT NULL,
    course_id TEXT NOT NULL,
    capacity INTEGER NOT NULL CHECK (capacity BETWEEN 2 AND 8),
    FOREIGN KEY (course_id) REFERENCES course (course_id)
);
```

The primary key identifies a group. Required values cannot be `NULL`. Capacity must be
between 2 and 8, inclusive, and the course must already exist.

### Practice

Design constraints for
`study_group_member(group_id, student_id, member_role)`. Prevent duplicate membership and
references to missing groups or students.

### `DELETE`, `DROP`, and `ALTER`

`DELETE` removes selected rows while retaining the table. `DROP TABLE` removes the table
and its data. `ALTER TABLE` changes a schema, with product-specific capabilities.

To remove only G01, use a `DELETE` statement with a predicate that identifies G01. Never
substitute `DROP TABLE` for a row-level removal.


## 2. Basic Queries

```sql
SELECT result_expressions
FROM input_relations
WHERE predicate;
```

For meaning, identify inputs, filter rows, and then determine output expressions. This is
not a claim about physical execution order.

### Worked Example

```sql
SELECT student_id, student_name
FROM student
WHERE dept_code = 'IM' AND student_id <> 'S101';
```

The result is `(S103, Kai Wu)`. Predict the rows if `AND` is changed to `OR`, and explain
which condition is true for each retained row.

### Duplicates and Expressions

SQL query results may contain duplicates:

```sql
SELECT dept_code FROM student;
SELECT DISTINCT dept_code FROM student;
```

The first result contains IM twice. The second returns DES, FIN, and IM. `DISTINCT`
applies to the complete result tuple.

```sql
SELECT course_id, credits, credits * 18 AS semester_hours
FROM course;
```

This expression calculates an output value; it does not modify stored credits.

### Multiple Inputs and Aliases

```sql
SELECT s.student_name, e.course_id, e.grade
FROM student AS s, enrollment AS e
WHERE s.student_id = e.student_id;
```

Without the matching predicate, four Student rows and six Enrollment rows produce 24
combinations. Chapter 4 replaces this older comma form with explicit `JOIN ... ON` syntax.


## 3. Patterns, Ranges, and Ordering

For `LIKE`, `%` matches zero or more characters and `_` matches exactly one character.

```sql
SELECT course_id, title
FROM course
WHERE title LIKE '%Technology%';
```

Case behavior depends on the DBMS and collation. `BETWEEN` includes both endpoints.

```sql
SELECT course_id, title, credits
FROM course
WHERE credits BETWEEN 1 AND 2
ORDER BY credits DESC, course_id ASC;
```

Without `ORDER BY`, do not rely on the displayed row order.

Practice: write one pattern for titles beginning with `Data` and another for titles whose
second character is `e`.


## 4. SQL Set Operations

Both query results must have the same number of columns with compatible types.

For DB students `{S101, S103}` and Financial Technology students `{S101, S102}`:

| Operation | Duplicate behavior | Result |
|---|---|---|
| `UNION` | Removes duplicates | S101, S102, S103 |
| `UNION ALL` | Retains all copies | S101, S101, S102, S103 |
| `INTERSECT` | Removes duplicates | S101 |
| `EXCEPT` | Left result minus right result | S103 |

Exchange the two sides of `EXCEPT` and predict the result. SQLite supports these forms
but not `INTERSECT ALL` or `EXCEPT ALL`.


## 5. `NULL` and Three-Valued Logic

`NULL` is not zero or an empty string. Most comparisons with `NULL` return `UNKNOWN`.

```sql
WHERE grade = NULL       -- incorrect test
WHERE grade IS NULL      -- correct test
```

`WHERE` retains only rows for which its predicate is `TRUE`; both `FALSE` and `UNKNOWN`
are removed.

| Expression | Result |
|---|---|
| `TRUE AND UNKNOWN` | `UNKNOWN` |
| `FALSE AND UNKNOWN` | `FALSE` |
| `TRUE OR UNKNOWN` | `TRUE` |
| `FALSE OR UNKNOWN` | `UNKNOWN` |
| `NOT UNKNOWN` | `UNKNOWN` |

### Worked Example

If one grade is temporarily set to `NULL`, `COUNT(*)` still counts that enrollment while
`COUNT(grade)` does not. The predicate `grade <> 'F'` also removes the row because its
truth value is `UNKNOWN`.

Practice: calculate `grade <> 'F'` for grades `A`, `F`, and `NULL`, and identify which
rows pass `WHERE`.


## 6. Aggregation and Grouping

`COUNT(*)` counts rows. `COUNT(attribute)` counts non-`NULL` values. `MIN`, `MAX`, `SUM`,
and `AVG` ignore `NULL` inputs.

```sql
SELECT COUNT(*) AS course_count,
       MIN(credits) AS min_credits,
       MAX(credits) AS max_credits,
       SUM(credits) AS total_credits,
       AVG(credits) AS avg_credits
FROM course;
```

For credits 3, 3, 3, and 2, the result is count 4, minimum 2, maximum 3, total 11, and
average 2.75.

### `GROUP BY`

```sql
SELECT dept_code,
       COUNT(*) AS course_count,
       AVG(credits) AS avg_credits
FROM course
GROUP BY dept_code
ORDER BY dept_code;
```

Nonaggregate output attributes should appear in `GROUP BY`. SQLite may accept additional
columns and choose an arbitrary value, but this course does not use that nonportable
behavior.

### `WHERE` and `HAVING`

`WHERE` filters rows before grouping. `HAVING` filters groups after aggregation.

```sql
SELECT dept_code, COUNT(*) AS course_count
FROM course
WHERE credits >= 3
GROUP BY dept_code
HAVING COUNT(*) >= 2;
```

The result is `(IM, 2)`. Practice: for the requirement "count only courses of at least
three credits and retain departments whose average is above 2.5," place each condition in
the correct clause and explain why.


## 7. Selected Subqueries

### `IN`

```sql
SELECT student_id, student_name
FROM student
WHERE student_id IN (
    SELECT student_id
    FROM enrollment
    WHERE course_id = 'DB201'
);
```

Run the inner query first. It returns S101 and S103; the outer query then retrieves their
names.

### `EXISTS` and Correlation

```sql
SELECT s.student_id, s.student_name
FROM student AS s
WHERE EXISTS (
    SELECT 1
    FROM enrollment AS e
    WHERE e.student_id = s.student_id
      AND e.grade IN ('A', 'A-')
);
```

The inner query refers to the current outer Student row. `EXISTS` asks only whether at
least one row exists. To find students with no enrollment, use `NOT EXISTS` with the same
identifier-matching predicate.

### `NOT IN` with `NULL`

```sql
SELECT student_id
FROM student
WHERE student_id NOT IN ('S104', NULL);
```

This returns no rows because the `NULL` comparison can make the predicate `UNKNOWN`.
A `NOT EXISTS` query with an explicit equality predicate avoids that problem. `NOT IN`
is acceptable when the schema or query guarantees that the comparison set cannot contain
`NULL`; state that guarantee rather than memorizing an unconditional ban.

### Common Table Expression

```sql
WITH counts AS (
    SELECT course_id, COUNT(*) AS enrollment_count
    FROM enrollment
    GROUP BY course_id
)
SELECT course_id, enrollment_count
FROM counts
WHERE enrollment_count >= 2;
```

A CTE names a query result for the current statement. It does not imply that the DBMS
must store a temporary table.

Subqueries in `FROM`, scalar subqueries, `SOME`, `ALL`, `UNIQUE`, `LATERAL`, and formal
multiset algebra are extensions unless separately taught and practiced.


## 8. Data Modification

Before modifying data, identify the target relation, affected rows, and possible key,
reference, `NOT NULL`, or `CHECK` violations.

```sql
INSERT INTO student (student_id, email, student_name, dept_code)
VALUES ('S105', 'noah.lee@example.edu', 'Noah Lee', 'IM');

UPDATE student
SET dept_code = 'FIN'
WHERE student_id = 'S105';

DELETE FROM student
WHERE student_id = 'S105';
```

The department must exist before insertion. An `UPDATE` or `DELETE` without the intended
`WHERE` clause may affect every row or violate references. Use a `SELECT` with the same
predicate to verify target rows first. The lab uses a savepoint and restores its sample
changes.

Practice: predict how many rows `UPDATE course SET credits = 4` changes. Then add a
predicate that targets only DB201 and verify it with `SELECT`.


## Common Errors

1. Expecting `SELECT` to remove duplicates automatically.
2. Omitting a matching predicate between relations.
3. Relying on row order without `ORDER BY`.
4. Using `= NULL` or `<> NULL`.
5. Confusing row filtering in `WHERE` with group filtering in `HAVING`.
6. Selecting a nonaggregate column that is not a grouping column.
7. Using `NOT IN` without checking whether the comparison set can contain `NULL`.
8. Running an `UPDATE` or `DELETE` before verifying its target rows.
9. Treating a SQLite-specific behavior as an SQL standard rule.


## Classroom and Individual Evidence

Retain:

1. Predictions and actual results for the executable SQL lab cell.
2. One `NULL` error and its correction.
3. One grouped query with the role of `WHERE`, grouping, `HAVING`, and `SELECT` labeled.
4. One subquery with separately verified inner and outer results.
5. One modification completed and reversed inside a savepoint.

When comparing solutions, judge the requested result, `NULL` handling, valid grouping,
and evidence from sample data. Peer ranking does not directly determine a grade; the
individual corrected work is the retained evidence.


## Chapter Summary

DDL defines structure and constraints; DML reads or modifies rows. SQL retains duplicates
unless instructed otherwise. `NULL` introduces `UNKNOWN`, and `WHERE` retains only
`TRUE`. Aggregation forms and filters groups in distinct stages. Subqueries should be
checked from the inside out, and every modification should be preceded by a target-row
and constraint check. Chapter 4 introduces explicit joins, views, constraints, and basic
transaction statements.


## After-Class Continuation

Write one additional grouped query and one additional subquery using the course schema.
Before execution, predict the retained rows and possible effect of `NULL`; afterward,
record the result and correct any difference between the prediction and observation.


## Executable Notebook Lab

Predict before running each cell, then compare the output with your explanation.


In [1]:
import sqlite3

print(f"Python {__import__('sys').version.split()[0]}; SQLite {sqlite3.sqlite_version}")
connection = sqlite3.connect(":memory:", isolation_level=None)
connection.execute("PRAGMA foreign_keys = ON")


def run_sql_script(connection, script, max_rows=20):
    """Execute a SQLite script and display result-producing statements."""
    buffer = ""
    for raw_line in script.splitlines():
        stripped = raw_line.strip()
        if stripped.startswith(".print"):
            message = stripped[len(".print"):].strip().strip("\"'")
            print(f"\n{message}")
            continue
        buffer += raw_line + "\n"
        if not sqlite3.complete_statement(buffer):
            continue
        statement = buffer.strip()
        buffer = ""
        if not statement:
            continue
        cursor = connection.execute(statement)
        if cursor.description:
            columns = [column[0] for column in cursor.description]
            rows = cursor.fetchmany(max_rows + 1)
            print(" | ".join(columns))
            for row in rows[:max_rows]:
                print(" | ".join("NULL" if value is None else str(value) for value in row))
            if len(rows) > max_rows:
                print(f"... additional rows omitted after {max_rows}")
    remaining = "\n".join(
        line for line in buffer.splitlines() if not line.strip().startswith("--")
    ).strip()
    if remaining:
        raise ValueError("The embedded SQL ends with an incomplete statement.")


Python 3.12.13; SQLite 3.53.1


### Course-registration setup


In [2]:
SQL_1 = """PRAGMA foreign_keys = ON;

DROP TABLE IF EXISTS enrollment;
DROP TABLE IF EXISTS course;
DROP TABLE IF EXISTS student;
DROP TABLE IF EXISTS department;

CREATE TABLE department (
    dept_code TEXT PRIMARY KEY,
    dept_name TEXT NOT NULL UNIQUE,
    building TEXT NOT NULL
);

CREATE TABLE student (
    student_id TEXT PRIMARY KEY,
    email TEXT NOT NULL UNIQUE,
    student_name TEXT NOT NULL,
    dept_code TEXT NOT NULL,
    FOREIGN KEY (dept_code) REFERENCES department (dept_code)
);

CREATE TABLE course (
    course_id TEXT PRIMARY KEY,
    title TEXT NOT NULL,
    dept_code TEXT NOT NULL,
    credits INTEGER NOT NULL CHECK (credits BETWEEN 1 AND 6),
    FOREIGN KEY (dept_code) REFERENCES department (dept_code)
);

CREATE TABLE enrollment (
    student_id TEXT NOT NULL,
    course_id TEXT NOT NULL,
    term TEXT NOT NULL,
    grade TEXT,
    PRIMARY KEY (student_id, course_id, term),
    FOREIGN KEY (student_id) REFERENCES student (student_id),
    FOREIGN KEY (course_id) REFERENCES course (course_id)
);

INSERT INTO department (dept_code, dept_name, building) VALUES
    ('DES', 'Digital Design', 'Hong Hall'),
    ('FIN', 'Finance', 'Cheng Hall'),
    ('IM', 'Information Management', 'Hong Hall');

INSERT INTO student (student_id, email, student_name, dept_code) VALUES
    ('S101', 'an.chen@example.edu', 'An Chen', 'IM'),
    ('S102', 'bea.lin@example.edu', 'Bea Lin', 'FIN'),
    ('S103', 'kai.wu@example.edu', 'Kai Wu', 'IM'),
    ('S104', 'mira.ho@example.edu', 'Mira Ho', 'DES');

INSERT INTO course (course_id, title, dept_code, credits) VALUES
    ('DB201', 'Database Management', 'IM', 3),
    ('FT210', 'Financial Technology', 'FIN', 3),
    ('ML230', 'Machine Learning', 'IM', 3),
    ('WD120', 'Web Design', 'DES', 2);

INSERT INTO enrollment (student_id, course_id, term, grade) VALUES
    ('S101', 'DB201', '115-1', 'A'),
    ('S101', 'FT210', '115-1', 'B+'),
    ('S102', 'FT210', '115-1', 'A-'),
    ('S103', 'DB201', '115-1', 'B'),
    ('S103', 'ML230', '115-1', 'A'),
    ('S104', 'WD120', '115-1', 'A-');
"""


In [3]:
run_sql_script(connection, SQL_1)


### Introduction to SQL lab


In [4]:
SQL_2 = """-- Chapter 3 executable examples
-- DBMS used for this file: SQLite 3
-- First run ../ch02_relational_model/course_registration_setup.sql.
-- Run this file from top to bottom. Modification examples use savepoints and
-- restore the base data before the script ends.

-- Example 1: DDL with keys and constraints.
DROP TABLE IF EXISTS study_group_member;
DROP TABLE IF EXISTS study_group;

CREATE TABLE study_group (
    group_id TEXT PRIMARY KEY,
    group_name TEXT NOT NULL,
    course_id TEXT NOT NULL,
    capacity INTEGER NOT NULL CHECK (capacity BETWEEN 2 AND 8),
    FOREIGN KEY (course_id) REFERENCES course (course_id)
);

CREATE TABLE study_group_member (
    group_id TEXT NOT NULL,
    student_id TEXT NOT NULL,
    member_role TEXT NOT NULL,
    PRIMARY KEY (group_id, student_id),
    FOREIGN KEY (group_id) REFERENCES study_group (group_id),
    FOREIGN KEY (student_id) REFERENCES student (student_id)
);

INSERT INTO study_group VALUES ('G01', 'SQL Practice', 'DB201', 4);
INSERT INTO study_group_member VALUES ('G01', 'S101', 'coordinator');
INSERT INTO study_group_member VALUES ('G01', 'S103', 'member');

SELECT group_id, group_name, course_id, capacity
FROM study_group;

-- Example 2: SELECT, FROM, aliases, and deterministic display order.
SELECT student_id, student_name AS name, dept_code
FROM student
ORDER BY student_id;

-- Example 3: SQL retains duplicates unless DISTINCT is requested.
SELECT dept_code
FROM student
ORDER BY dept_code;

SELECT DISTINCT dept_code
FROM student
ORDER BY dept_code;

-- Example 4: expressions do not update stored data.
SELECT course_id, title, credits, credits * 18 AS semester_hours
FROM course
ORDER BY course_id;

-- Example 5: WHERE with comparisons and logical connectives.
SELECT student_id, student_name
FROM student
WHERE dept_code = 'IM' AND student_id <> 'S101'
ORDER BY student_id;

-- Example 6: multiple relations and a matching predicate.
-- Explicit JOIN syntax is introduced in Chapter 4. This Chapter 3 form is used
-- to connect FROM, WHERE, and relational-algebra product/selection concepts.
SELECT s.student_name, e.course_id, e.grade
FROM student AS s, enrollment AS e
WHERE s.student_id = e.student_id
ORDER BY s.student_id, e.course_id;

-- Example 7: LIKE, BETWEEN, and ORDER BY.
SELECT course_id, title, credits
FROM course
WHERE title LIKE '%Technology%' OR credits BETWEEN 1 AND 2
ORDER BY credits DESC, course_id ASC;

-- Example 8a: UNION removes duplicates.
SELECT student_id FROM enrollment WHERE course_id = 'DB201'
UNION
SELECT student_id FROM enrollment WHERE course_id = 'FT210'
ORDER BY student_id;

-- Example 8b: UNION ALL retains copies from both inputs.
SELECT student_id FROM enrollment WHERE course_id = 'DB201'
UNION ALL
SELECT student_id FROM enrollment WHERE course_id = 'FT210'
ORDER BY student_id;

-- Example 8c: INTERSECT and EXCEPT.
SELECT student_id FROM enrollment WHERE course_id = 'DB201'
INTERSECT
SELECT student_id FROM enrollment WHERE course_id = 'FT210'
ORDER BY student_id;

SELECT student_id FROM enrollment WHERE course_id = 'DB201'
EXCEPT
SELECT student_id FROM enrollment WHERE course_id = 'FT210'
ORDER BY student_id;

-- Example 9: NULL, three-valued logic, and aggregate handling.
SAVEPOINT null_demo;
UPDATE enrollment
SET grade = NULL
WHERE student_id = 'S102' AND course_id = 'FT210' AND term = '115-1';

SELECT student_id, course_id
FROM enrollment
WHERE grade IS NULL
ORDER BY student_id, course_id;

-- This intentionally returns no rows: grade = NULL is unknown, not true.
SELECT student_id, course_id
FROM enrollment
WHERE grade = NULL;

SELECT COUNT(*) AS enrollment_rows,
       COUNT(grade) AS known_grades
FROM enrollment;

ROLLBACK TO null_demo;
RELEASE null_demo;

-- Example 10: aggregate functions.
SELECT COUNT(*) AS course_count,
       MIN(credits) AS min_credits,
       MAX(credits) AS max_credits,
       SUM(credits) AS total_credits,
       AVG(credits) AS avg_credits
FROM course;

-- Example 11: GROUP BY creates one result row per group.
SELECT dept_code, COUNT(*) AS course_count, AVG(credits) AS avg_credits
FROM course
GROUP BY dept_code
ORDER BY dept_code;

-- Example 12: WHERE filters rows before grouping; HAVING filters groups.
SELECT dept_code, COUNT(*) AS course_count
FROM course
WHERE credits >= 3
GROUP BY dept_code
HAVING COUNT(*) >= 2
ORDER BY dept_code;

-- Example 13: IN subquery.
SELECT student_id, student_name
FROM student
WHERE student_id IN (
    SELECT student_id
    FROM enrollment
    WHERE course_id = 'DB201'
)
ORDER BY student_id;

-- Example 14: correlated EXISTS subquery.
SELECT s.student_id, s.student_name
FROM student AS s
WHERE EXISTS (
    SELECT 1
    FROM enrollment AS e
    WHERE e.student_id = s.student_id
      AND e.grade IN ('A', 'A-')
)
ORDER BY s.student_id;

-- Example 15: NOT IN with a NULL in the subquery returns no rows.
WITH blocked(student_id) AS (
    VALUES ('S104'), (NULL)
)
SELECT student_id
FROM student
WHERE student_id NOT IN (SELECT student_id FROM blocked)
ORDER BY student_id;

-- Example 16: NOT EXISTS expresses the intended exclusion despite the NULL.
WITH blocked(student_id) AS (
    VALUES ('S104'), (NULL)
)
SELECT s.student_id
FROM student AS s
WHERE NOT EXISTS (
    SELECT 1
    FROM blocked AS b
    WHERE b.student_id = s.student_id
)
ORDER BY s.student_id;

-- Example 17: subquery in FROM.
SELECT course_id, enrollment_count
FROM (
    SELECT course_id, COUNT(*) AS enrollment_count
    FROM enrollment
    GROUP BY course_id
) AS counts
WHERE enrollment_count >= 2
ORDER BY course_id;

-- Example 18: the same intermediate result expressed with WITH.
WITH counts AS (
    SELECT course_id, COUNT(*) AS enrollment_count
    FROM enrollment
    GROUP BY course_id
)
SELECT course_id, enrollment_count
FROM counts
WHERE enrollment_count >= 2
ORDER BY course_id;

-- Example 19: scalar correlated subquery.
SELECT c.course_id,
       c.title,
       (
           SELECT COUNT(*)
           FROM enrollment AS e
           WHERE e.course_id = c.course_id
       ) AS enrollment_count
FROM course AS c
ORDER BY c.course_id;

-- Example 20: INSERT, UPDATE, and DELETE inside a reversible demonstration.
SAVEPOINT dml_demo;

INSERT INTO student (student_id, email, student_name, dept_code)
VALUES ('S105', 'noah.lee@example.edu', 'Noah Lee', 'IM');

UPDATE student
SET dept_code = 'FIN'
WHERE student_id = 'S105';

SELECT student_id, student_name, dept_code
FROM student
WHERE student_id = 'S105';

DELETE FROM student
WHERE student_id = 'S105';

SELECT COUNT(*) AS remaining_s105
FROM student
WHERE student_id = 'S105';

ROLLBACK TO dml_demo;
RELEASE dml_demo;

-- Student practice. Predict before executing your own statements.
-- P1. Return course_id and title for all three-credit courses.
-- P2. Return distinct department codes for students, in descending order.
-- P3. Count enrollments for each course and retain groups with at least two.
-- P4. Use EXISTS to find students enrolled in FT210.
-- P5. Inside a savepoint, insert a valid course and then restore the database.
-- P6. Explain why WHERE grade <> 'F' does not retain a NULL grade.
"""


In [5]:
run_sql_script(connection, SQL_2)


group_id | group_name | course_id | capacity
G01 | SQL Practice | DB201 | 4
student_id | name | dept_code
S101 | An Chen | IM
S102 | Bea Lin | FIN
S103 | Kai Wu | IM
S104 | Mira Ho | DES
dept_code
DES
FIN
IM
IM
dept_code
DES
FIN
IM
course_id | title | credits | semester_hours
DB201 | Database Management | 3 | 54
FT210 | Financial Technology | 3 | 54
ML230 | Machine Learning | 3 | 54
WD120 | Web Design | 2 | 36
student_id | student_name
S103 | Kai Wu
student_name | course_id | grade
An Chen | DB201 | A
An Chen | FT210 | B+
Bea Lin | FT210 | A-
Kai Wu | DB201 | B
Kai Wu | ML230 | A
Mira Ho | WD120 | A-
course_id | title | credits
FT210 | Financial Technology | 3
WD120 | Web Design | 2
student_id
S101
S102
S103
student_id
S101
S101
S102
S103
student_id
S101
student_id
S103
student_id | course_id
S102 | FT210
student_id | course_id
enrollment_rows | known_grades
6 | 5
course_count | min_credits | max_credits | total_credits | avg_credits
4 | 2 | 3 | 11 | 2.75
dept_code | course_count | avg

### Reproducibility Check


In [6]:
assert connection.execute("SELECT COUNT(*) FROM department").fetchone()[0] == 3
assert connection.execute("SELECT COUNT(*) FROM student").fetchone()[0] == 4
assert connection.execute("SELECT COUNT(*) FROM course").fetchone()[0] == 4
assert connection.execute("SELECT COUNT(*) FROM enrollment").fetchone()[0] == 6
assert connection.execute("PRAGMA foreign_key_check").fetchall() == []
print("Notebook checks passed.")


Notebook checks passed.


In [7]:
connection.close()
print("In-memory database closed.")


In-memory database closed.
